# Lezione 10 — RAG: un chatbot che risponde sui tuoi documenti

## Obiettivi di apprendimento

- Capire cos'è RAG (*Retrieval-Augmented Generation*) e quale problema risolve.
- Riconoscere i cinque pezzi di una pipeline RAG: loader, text splitter,
  embeddings, vector store e retriever + chain.
- Costruire, con LangChain, Chroma e modelli gratuiti di Hugging Face, un
  chatbot che risponde a domande su un PDF proprio.
- Eseguire l'intera pipeline su Colab senza pagare nulla né dipendere da OpenAI.

## Cosa costruiremo

Un "Personal Data Assistant": generiamo un PDF di esempio con
`reportlab`, lo indicizziamo con embeddings e Chroma, e gli facciamo domande
sul suo contenuto usando un LLM gratuito. È la stessa architettura del
progetto `wbphl-build_own_chatbot_without_open_ai` di IBM, nella sua variante
100% gratuita (`worker_huggingFace.py`).

## Appunti: cos'è RAG e come funziona

Un LLM generico non sa nulla di un documento che non ha visto durante il suo
addestramento. Riaddestrarlo ogni volta che compare un documento nuovo è
carissimo. **RAG** risolve questo problema senza toccare il modello: invece di
dargli in pasto l'intero documento, cerca prima i frammenti rilevanti e glieli
consegna come contesto appena prima della domanda — come dare a uno stagista solo le
pagine esatte del manuale di cui ha bisogno, invece di tutto il libro.

La pipeline ha cinque pezzi:

1. **Document loader**: carica il documento (qui, `PyPDFLoader` per il PDF).
2. **Text splitter**: spezzetta il testo in frammenti gestibili
   (`RecursiveCharacterTextSplitter`, con sovrapposizione tra frammenti).
3. **Embeddings**: converte ogni frammento in un vettore di significato
   (`sentence-transformers/all-MiniLM-L6-v2`).
4. **Vector store**: salva questi vettori per cercare per similarità
   (**Chroma**).
5. **Retriever + chain**: cerca i frammenti più simili alla domanda e
   li passa all'LLM (`RetrievalQA`, di LangChain).

Nel materiale originale di IBM, l'LLM è `WatsonxLLM` (a pagamento). Qui
usiamo la vera alternativa gratuita dello stesso repository di IBM,
`HuggingFaceHub` con il modello `tiiuae/falcon-7b-instruct` — serve solo un
token gratuito di Hugging Face, senza carta di credito.

In [ ]:
# Installiamo le librerie necessarie: LangChain e le sue integrazioni,
# Chroma come database vettoriale, sentence-transformers per gli
# embeddings, pypdf per leggere i PDF, e reportlab per generare il nostro PDF di
# esempio.
!pip install -q langchain langchain-community chromadb sentence-transformers pypdf reportlab huggingface_hub

### Passo 0: creiamo un PDF di esempio

In un'app reale, questo PDF lo caricherebbe l'utente (il suo manuale, contratto,
report, ecc.). Qui ne generiamo uno noi stessi con `reportlab` affinché
il notebook sia 100% autocontenuto: una piccola politica di ferie
fittizia di un'azienda, con dati concreti che poi chiederemo
al chatbot.

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def crear_pdf_ejemplo(ruta="documento_ejemplo.pdf"):
    contenido = """Manuale interno di Acme Corp - Politica delle ferie e dei viaggi

    Articolo 1. Giorni di ferie.
    Tutti i dipendenti a tempo pieno di Acme Corp hanno diritto a 22
    giorni lavorativi di ferie retribuite per ogni anno completo lavorato.
    I dipendenti a tempo parziale ricevono giorni proporzionali al loro orario.

    Articolo 2. Richiesta di ferie.
    Le ferie devono essere richieste con almeno 15 giorni di anticipo
    tramite il portale interno delle Risorse Umane. Il responsabile diretto
    deve approvare la richiesta entro un termine massimo di 5 giorni lavorativi.

    Articolo 3. Rimborso spese di viaggio.
    I dipendenti che viaggiano per motivi di lavoro possono richiedere il
    rimborso di trasporto, alloggio e pasti presentando le ricevute
    originali entro i 30 giorni successivi al viaggio. Il limite giornaliero
    per i pasti e di 40 dollari.

    Articolo 4. Giorni di malattia.
    I dipendenti hanno diritto a 10 giorni retribuiti per malattia all'anno,
    che non richiedono certificato medico se sono meno di 3 giorni consecutivi.
    """
    c = canvas.Canvas(ruta, pagesize=letter)
    texto = c.beginText(40, 750)
    texto.setFont("Helvetica", 11)
    for linea in contenido.strip().split("\n"):
        texto.textLine(linea.strip())
    c.drawText(texto)
    c.save()
    return ruta

ruta_pdf = crear_pdf_ejemplo()
print("PDF di esempio creato in:", ruta_pdf)

### Passo 1: l'LLM e gli embeddings gratuiti

`HuggingFaceHub` richiede un token gratuito di Hugging Face (te lo chiede una
sola volta). Puoi crearne uno su https://huggingface.co/settings/tokens con
un account gratuito — non serve carta di credito.

In [ ]:
import os
from getpass import getpass

if "HUGGINGFACEHUB_API_TOKEN" not in os.environ:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass(
        "Incolla il tuo token gratuito di Hugging Face (https://huggingface.co/settings/tokens): "
    )

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFaceHub

# Modello di embeddings: converte il testo in vettori di significato.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

# LLM gratuito di Hugging Face (vera alternativa a WatsonxLLM nel
# materiale di IBM: worker_huggingFace.py usa esattamente questo modello).
llm = HuggingFaceHub(
    repo_id="tiiuae/falcon-7b-instruct",
    model_kwargs={"temperature": 0.5, "max_new_tokens": 200},
)

### Passo 2: caricare il PDF e spezzettarlo

`PyPDFLoader` estrae il testo dal PDF. `RecursiveCharacterTextSplitter` lo
divide in frammenti di 1024 caratteri, con 64 caratteri di sovrapposizione tra
frammenti consecutivi — gli stessi valori usati dal `worker.py` reale di
IBM.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = PyPDFLoader(ruta_pdf)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=64)
texts = text_splitter.split_documents(documents)

print(f"Il PDF e stato diviso in {len(texts)} frammenti.")
print("--- Primo frammento ---")
print(texts[0].page_content[:300])

### Passo 3: indicizzare in Chroma e costruire la chain RAG

Salviamo i frammenti (convertiti in vettori) in Chroma, e costruiamo
un `retriever` che cerca i 6 frammenti più rilevanti per ogni domanda
(`search_type="mmr"` evita che questi 6 frammenti siano tutti quasi identici
tra loro). `RetrievalQA` con `chain_type="stuff"` inserisce questi frammenti
direttamente nel prompt dell'LLM.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA

db = Chroma.from_documents(texts, embeddings)

retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25})

chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever, input_key="question"
)

### Passo 4: chiediamo al chatbot informazioni sul nostro PDF

Proprio come in `process_prompt` del `worker.py` reale, salviamo la cronologia
della conversazione come una lista di coppie (domanda, risposta).

In [ ]:
chat_history = []

def process_prompt(prompt):
    output = chain({"question": prompt, "chat_history": chat_history})
    answer = output["result"]
    chat_history.append((prompt, answer))
    return answer

respuesta = process_prompt("Quanti giorni di ferie ho all'anno?")
print(respuesta)

In [ ]:
respuesta = process_prompt("Qual e il limite giornaliero per i pasti in un viaggio di lavoro?")
print(respuesta)

## Riferimento: la versione con watsonx (a pagamento) del materiale originale

```python
# worker.py (frammento reale, richiede credenziali watsonx)
from langchain_ibm import WatsonxLLM

def init_llm():
    params = {
        GenParams.MAX_NEW_TOKENS: 256,
        GenParams.TEMPERATURE: 0.1,
    }
    llm = WatsonxLLM(
        model_id="meta-llama/llama-4-maverick-17b-128e-instruct-fp8",
        url="https://us-south.ml.cloud.ibm.com",
        params=params,
        project_id=project_id,
    )
    embeddings = HuggingFaceInstructEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": DEVICE},
    )
    return llm, embeddings
```

L'unica differenza reale sta in come viene creato l'LLM (`WatsonxLLM` invece di
`HuggingFaceHub`); il resto della pipeline — caricare, spezzettare, indicizzare,
recuperare, rispondere — è identico a quello che hai appena eseguito.

## Riepilogo della lezione

- RAG permette a un LLM di rispondere su documenti che non ha mai visto nel suo
  addestramento, senza riaddestrarlo: recupera frammenti rilevanti e glieli
  consegna come contesto.
- La pipeline: document loader → text splitter → embeddings → vector store
  → retriever + chain.
- `RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=64)` e
  `Chroma` sono i pezzi esatti usati dal materiale reale di IBM.
- Con `HuggingFaceHub` (`falcon-7b-instruct`) e `HuggingFaceEmbeddings`
  (`all-MiniLM-L6-v2`) puoi montare l'intera pipeline gratis, senza OpenAI né
  watsonx.

Continua con:
- Capitolo del manuale: `manual/14_rag_documentos.md`
- Notebook di pratica: `notebooks_practica/clase10_practica_rag_documentos.ipynb`
  (lì riprodurrai i TODO reali di `worker.py`/`Worker_completed.py`).